# Qwen3-ASR Colab A100 Evaluation Lab

Repo-friendly notebook for testing `Qwen/Qwen3-ASR-1.7B` on long Chinese, English, and Chinese-English mixed audio.

This version assumes:

- public GitHub repo, so **no `GH_TOKEN`**;
- Hugging Face model downloads via **Colab Secret `HF_TOKEN`**;
- audio files manually uploaded to Google Drive;
- optional open-source ASR baseline via `faster-whisper`;
- optional **text-only** ASR output triage via `google.colab.ai`.

Important: `google.colab.ai` only supports text-to-text generation. It cannot listen to audio and is not a true ASR judge. Use it to flag suspicious chunks, not to prove transcript faithfulness.

In [ ]:
#@title Runtime check
import os, sys, subprocess, json, textwrap, pathlib, time, random, re, math, shutil
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not importable yet:", repr(e))

subprocess.run(["bash", "-lc", "nvidia-smi || true"], check=False)

## 1. Mount Drive and clone/install the public repo

After you push this starter repo, set `PUBLIC_REPO_URL` to your public GitHub repo URL. No GitHub token is needed.

Drive is used for audio inputs and persistent output runs. `/content` is used for active compute.

In [ ]:
#@title Startup config: repo + Drive paths
from pathlib import Path
import os, subprocess, sys, time, json

MOUNT_DRIVE = True  #@param {type:"boolean"}
PUBLIC_REPO_URL = "https://github.com/liuwen/qwen-asr-eval.git"  #@param {type:"string"}
PUBLIC_REPO_BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/qwen-asr-eval"  #@param {type:"string"}
DRIVE_REPO_DIR = ""  #@param {type:"string"}

DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/asr/qwen-asr-eval"  #@param {type:"string"}
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/asr/audio"  #@param {type:"string"}

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped/failed:", repr(e))

repo_dir = Path(REPO_DIR)
if PUBLIC_REPO_URL.strip():
    if repo_dir.exists():
        print("Repo dir already exists; pulling latest:", repo_dir)
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", PUBLIC_REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", PUBLIC_REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", PUBLIC_REPO_BRANCH, PUBLIC_REPO_URL, str(repo_dir)], check=True)
elif DRIVE_REPO_DIR.strip():
    repo_dir = Path(DRIVE_REPO_DIR)
    assert repo_dir.exists(), f"DRIVE_REPO_DIR not found: {repo_dir}"
elif repo_dir.exists():
    print("Using existing local repo dir:", repo_dir)
else:
    raise RuntimeError("Set PUBLIC_REPO_URL after pushing this repo, or set DRIVE_REPO_DIR / REPO_DIR to an existing checkout.")

print("Repo dir:", repo_dir)
print("Install happens in the next cell after system dependencies are ready.")

## 2. Install Colab dependencies

This uses the repo's `requirements-colab.txt`. `torch` normally comes from Colab's runtime image, so the requirements avoid forcing a torch reinstall.

In [ ]:
%%capture
#@title Install system and Python packages
import subprocess, sys
from pathlib import Path

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "jq"], check=True)

req = Path(repo_dir) / "requirements-colab.txt"
assert req.exists(), f"Missing requirements file: {req}"
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-r", str(req)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_dir), "--no-deps"], check=True)

## 3. Hugging Face authentication

Add `HF_TOKEN` in Colab Secrets before running this cell. This notebook intentionally does not use `GH_TOKEN` or `GEMINI_API_KEY`.

In [ ]:
#@title Read HF_TOKEN from Colab Secrets
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab userdata unavailable:", repr(e))

REQUIRE_HF_TOKEN = True  #@param {type:"boolean"}
if REQUIRE_HF_TOKEN and not HF_TOKEN:
    raise RuntimeError("Missing Colab Secret: HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF_TOKEN configured for Hugging Face downloads.")
    except Exception as e:
        print("Hugging Face login failed, but env token is still set:", repr(e))
else:
    print("No HF_TOKEN configured. Public model downloads may still work, but rate limits may be worse.")

## 4. Run configuration

Choose a use case and an audio file. If `AUDIO_PATH` is empty, the notebook scans `DRIVE_AUDIO_DIR` using `AUDIO_GLOB` and picks `AUDIO_FILE_INDEX`.

Use cases:

- `zh`: force Chinese for Qwen and Whisper.
- `en`: force English.
- `mixed` / `auto`: let models detect language.

In [ ]:
#@title ASR + audio config
from pathlib import Path
from datetime import datetime
import json, os

USE_CASE = "auto"  #@param ["auto", "zh", "en", "mixed"]
AUDIO_PATH = ""  #@param {type:"string"}
AUDIO_GLOB = "*"  #@param {type:"string"}
AUDIO_FILE_INDEX = 0  #@param {type:"integer"}

ASR_MODEL = "Qwen/Qwen3-ASR-1.7B"  #@param {type:"string"}
USE_FORCED_ALIGNER = False  #@param {type:"boolean"}
ALIGNER_MODEL = "Qwen/Qwen3-ForcedAligner-0.6B"  #@param {type:"string"}

CHUNK_SECONDS = 300  #@param {type:"integer"}
CHUNK_OVERLAP_SECONDS = 5  #@param {type:"integer"}
QWEN_BATCH_SIZE = 1  #@param {type:"integer"}
QWEN_MAX_INFERENCE_BATCH_SIZE = 4  #@param {type:"integer"}
QWEN_MAX_NEW_TOKENS = 4096  #@param {type:"integer"}

RUN_WHISPER_BASELINE = False  #@param {type:"boolean"}
WHISPER_MODEL_SIZE = "large-v3"  #@param ["large-v3", "medium", "small"]

RUN_COLAB_AI_TEXT_JUDGE = True  #@param {type:"boolean"}
COLAB_AI_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
MAX_TEXT_JUDGE_CHUNKS = 6  #@param {type:"integer"}

REFERENCE_TEXT_PATH = ""  #@param {type:"string"}

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_ROOT = Path(DRIVE_PROJECT_ROOT)
RUN_DIR = PROJECT_ROOT / "runs" / run_id
WORK_DIR = RUN_DIR / "work"
OUT_DIR = RUN_DIR / "outputs"
CHUNK_DIR = WORK_DIR / "chunks"
for p in [PROJECT_ROOT, RUN_DIR, WORK_DIR, OUT_DIR, CHUNK_DIR, Path(DRIVE_AUDIO_DIR)]:
    p.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)
print("OUT_DIR:", OUT_DIR)

### Optional: download Xiaoyuzhou eval audio artifacts\n\nThis resolves podcast episode URLs through RSSHub, downloads the audio enclosure only if it is not already present, and normalizes to 16 kHz mono WAV. The first default is the short Mandarin-English sample for smoke testing the full workflow.\n\nUse downloaded audio for private ASR evaluation only; do not redistribute source audio or full generated transcripts unless the publisher permits it.

In [ ]:
#@title Download Xiaoyuzhou eval audio if missing
from pathlib import Path
import json

from asr_eval.xiaoyuzhou import (
    XIAOYUZHOU_EVAL_MANIFEST,
    fetch_eval_audio,
    select_manifest_items,
)

DOWNLOAD_EVAL_AUDIO = True  #@param {type:"boolean"}
# Comma-separated IDs. Default is the shortest mixed Mandarin-English item for smoke tests.
EVAL_AUDIO_IDS = "mixed_zh_en_wujimacha_22e01"  #@param {type:"string"}
RSSHUB_BASE_URL = "https://rsshub.app"  #@param {type:"string"}
SET_AUDIO_PATH_TO_FIRST_DOWNLOAD = True  #@param {type:"boolean"}

print("Available eval ids:")
for item in XIAOYUZHOU_EVAL_MANIFEST:
    print(f"- {item['id']} [{item['scenario']}, {item['duration_min']} min]")

if DOWNLOAD_EVAL_AUDIO:
    requested_ids = [x.strip() for x in EVAL_AUDIO_IDS.split(",") if x.strip()]
    eval_items = select_manifest_items(requested_ids)
    downloaded_eval_audio = fetch_eval_audio(
        eval_items,
        audio_dir=DRIVE_AUDIO_DIR,
        rsshub_base=RSSHUB_BASE_URL,
    )
    print(json.dumps(downloaded_eval_audio, ensure_ascii=False, indent=2)[:4000])
    if SET_AUDIO_PATH_TO_FIRST_DOWNLOAD and downloaded_eval_audio and not AUDIO_PATH.strip():
        AUDIO_PATH = downloaded_eval_audio[0]["normalized_wav_path"]
        print("AUDIO_PATH set to:", AUDIO_PATH)
else:
    downloaded_eval_audio = []
    print("DOWNLOAD_EVAL_AUDIO=False; using AUDIO_PATH or DRIVE_AUDIO_DIR discovery.")


In [ ]:
#@title Pick audio file from Drive or explicit path
from asr_eval.audio import discover_audio_files, ffprobe_duration, fmt_ts
from pathlib import Path

if AUDIO_PATH.strip():
    audio_path = Path(AUDIO_PATH).expanduser()
else:
    candidates = discover_audio_files(DRIVE_AUDIO_DIR, patterns=[AUDIO_GLOB])
    print(f"Found {len(candidates)} candidate audio files under {DRIVE_AUDIO_DIR!r}")
    for i, p in enumerate(candidates[:50]):
        print(f"[{i:02d}] {p}")
    if not candidates:
        raise FileNotFoundError(f"No audio files found in {DRIVE_AUDIO_DIR} with AUDIO_GLOB={AUDIO_GLOB!r}")
    audio_path = candidates[int(AUDIO_FILE_INDEX)]

assert audio_path.exists(), f"Audio file not found: {audio_path}"
print("Selected audio:", audio_path)
try:
    print("Duration:", fmt_ts(ffprobe_duration(audio_path)))
except Exception as e:
    print("Could not probe duration yet:", repr(e))

## 5. Normalize and chunk audio

The normalized working file is 16 kHz mono PCM WAV. This is larger than MP3/M4A but stable and reproducible. For long podcasts, chunking is the operational control that matters most.

In [ ]:
#@title Normalize to 16 kHz mono WAV and split into chunks
from asr_eval.audio import normalize_to_wav, chunk_wav, write_jsonl, ffprobe_duration, fmt_ts
import json

NORMALIZED_WAV = normalize_to_wav(audio_path, WORK_DIR / "normalized_16k_mono.wav")
print("Normalized:", NORMALIZED_WAV)
print("Normalized duration:", fmt_ts(ffprobe_duration(NORMALIZED_WAV)))

chunks = chunk_wav(
    NORMALIZED_WAV,
    CHUNK_DIR,
    chunk_seconds=CHUNK_SECONDS,
    overlap_seconds=CHUNK_OVERLAP_SECONDS,
)
write_jsonl(OUT_DIR / "chunks_manifest.jsonl", chunks)
print("Chunks:", len(chunks))
print(json.dumps(chunks[:3], ensure_ascii=False, indent=2))

## 6. Load and run Qwen3-ASR

Start without forced alignment. Add forced alignment only when you need timestamps and after the basic ASR run succeeds.

In [ ]:
#@title Load Qwen3-ASR
from asr_eval.qwen_runner import load_qwen_model, qwen_language_for_use_case

QWEN_LANGUAGE = qwen_language_for_use_case(USE_CASE)
print("USE_CASE:", USE_CASE)
print("QWEN_LANGUAGE:", QWEN_LANGUAGE)

qwen_model = load_qwen_model(
    ASR_MODEL,
    max_inference_batch_size=QWEN_MAX_INFERENCE_BATCH_SIZE,
    max_new_tokens=QWEN_MAX_NEW_TOKENS,
    use_forced_aligner=USE_FORCED_ALIGNER,
    forced_aligner_model=ALIGNER_MODEL,
)
print("Loaded:", ASR_MODEL)

In [ ]:
#@title Transcribe chunks with Qwen3-ASR
from asr_eval.qwen_runner import transcribe_chunks
from asr_eval.audio import write_jsonl
from asr_eval.reporting import save_transcript
import pandas as pd

qwen_rows = transcribe_chunks(
    qwen_model,
    chunks,
    language=QWEN_LANGUAGE,
    batch_size=QWEN_BATCH_SIZE,
    return_time_stamps=USE_FORCED_ALIGNER,
    model_name=ASR_MODEL,
)
qwen_df = pd.DataFrame(qwen_rows)
write_jsonl(OUT_DIR / "qwen_chunks.jsonl", qwen_rows)
qwen_df.drop(columns=["time_stamps"], errors="ignore").to_csv(OUT_DIR / "qwen_chunks.csv", index=False)
qwen_saved = save_transcript(OUT_DIR, "qwen", "Qwen3-ASR transcript", qwen_rows)
qwen_transcript = qwen_saved["text"]

print("Saved:", OUT_DIR / "qwen_chunks.jsonl")
print("Saved:", qwen_saved["md_path"])
display(qwen_df[["chunk_id", "start_ts", "end_ts", "detected_language", "text"]].head())
print(qwen_transcript[:2000])

## 7. Optional ASR baseline: faster-whisper

This is the audio-capable baseline. It is not Gemini and does not require an API key. It is optional because it adds model download and inference time.

In [ ]:
#@title Run faster-whisper baseline
import pandas as pd
from asr_eval.whisper_baseline import run_whisper_baseline
from asr_eval.audio import write_jsonl
from asr_eval.reporting import save_transcript

if RUN_WHISPER_BASELINE:
    whisper_rows = run_whisper_baseline(
        chunks,
        model_size=WHISPER_MODEL_SIZE,
        use_case=USE_CASE,
        device="cuda",
        compute_type="float16",
    )
    whisper_df = pd.DataFrame(whisper_rows)
    write_jsonl(OUT_DIR / "whisper_chunks.jsonl", whisper_rows)
    whisper_df.drop(columns=["segments"], errors="ignore").to_csv(OUT_DIR / "whisper_chunks.csv", index=False)
    whisper_saved = save_transcript(OUT_DIR, "whisper", "faster-whisper transcript", whisper_rows)
    whisper_transcript = whisper_saved["text"]
    print("Saved:", OUT_DIR / "whisper_chunks.jsonl")
    print("Saved:", whisper_saved["md_path"])
    display(whisper_df[["chunk_id", "start_ts", "end_ts", "detected_language", "text"]].head())
else:
    whisper_rows = None
    whisper_df = None
    whisper_transcript = None
    print("RUN_WHISPER_BASELINE=False; skipping baseline.")

## 8. Reference metrics, if you have a transcript

For pure Chinese, CER is usually more useful. For pure English, WER is usually more familiar. For mixed Chinese-English, compute both but inspect samples manually.

In [ ]:
#@title Compute WER/CER if REFERENCE_TEXT_PATH is set
from pathlib import Path
from asr_eval.metrics import compute_metrics
import pandas as pd, json

metric_rows = []
if REFERENCE_TEXT_PATH.strip():
    ref_path = Path(REFERENCE_TEXT_PATH)
    assert ref_path.exists(), f"Reference text path not found: {ref_path}"
    reference_text = ref_path.read_text(encoding="utf-8")
    metric_rows.append({"system": "qwen", **compute_metrics(reference_text, qwen_transcript)})
    if whisper_transcript:
        metric_rows.append({"system": "whisper", **compute_metrics(reference_text, whisper_transcript)})

    metrics_df = pd.DataFrame(metric_rows)
    metrics_df.to_csv(OUT_DIR / "reference_metrics.csv", index=False)
    (OUT_DIR / "reference_metrics.json").write_text(json.dumps(metric_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    display(metrics_df)
else:
    print("REFERENCE_TEXT_PATH is empty; skipping reference-based metrics.")

## 9A. Resume a script-driven smoke run in the Colab UI — self-healing version

Use this section when `scripts/colab_smoke_xiaoyuzhou.py` already ran Qwen ASR from `colab exec`, but `google.colab.ai` failed there because Colab AI/userdata are only available from the web UI.

These cells mount Drive in this UI runtime, clone the repo if `/content/qwen-asr-eval` is missing, then auto-discover the latest `qwen_chunks.jsonl` under Drive. You do **not** need to rerun model download or ASR.

Version marker: `resume-ui-v3-auto-discover`.

In [ ]:
#@title Resume latest script-driven smoke output from Drive (auto-discover)
import sys, json, subprocess, os
from pathlib import Path

RESUME_CELL_VERSION = "resume-ui-v3-auto-discover"
print("Resume cell version:", RESUME_CELL_VERSION)

MOUNT_DRIVE_IN_UI = True  #@param {type:"boolean"}
REPO_DIR = Path("/content/qwen-asr-eval")  #@param {type:"string"}
PUBLIC_REPO_URL = "https://github.com/liuwen/qwen-asr-eval.git"  #@param {type:"string"}
PUBLIC_REPO_BRANCH = "main"  #@param {type:"string"}
RUNS_ROOT = Path("/content/drive/MyDrive/asr/qwen-asr-eval/runs")  #@param {type:"string"}
RUN_ID_OR_OUTPUT_DIR = ""  #@param {type:"string"}
AUTO_DISCOVER_UNDER_MYDRIVE = True  #@param {type:"boolean"}

my_drive = Path("/content/drive/MyDrive")
if MOUNT_DRIVE_IN_UI and not my_drive.exists():
    print("Drive is not mounted in this notebook runtime; mounting now...")
    from google.colab import drive
    drive.mount("/content/drive")

if not my_drive.exists():
    raise RuntimeError(
        "Drive is not mounted at /content/drive/MyDrive. "
        "Run this cell in the Colab web UI and authorize Drive access."
    )
print("Drive mounted:", my_drive)

# /content is ephemeral. Clone the public repo in this UI runtime if missing.
if not REPO_DIR.exists():
    print(f"Repo not found at {REPO_DIR}; cloning public repo...")
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", PUBLIC_REPO_BRANCH,
        PUBLIC_REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print(f"Repo exists at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Install small Python deps needed by the resume/judge cells. The Web UI runtime may be fresh
# even if the CLI runtime already installed them.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pandas"], check=True)

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Resolve OUT_DIR robustly. Accept either a run id, a run dir, or an outputs dir.
def _candidate_from_user_value(value: str):
    value = value.strip()
    if not value:
        return None
    p = Path(value)
    if not p.is_absolute():
        p = RUNS_ROOT / value
    if p.name == "outputs" and (p / "qwen_chunks.jsonl").exists():
        return p
    if (p / "outputs" / "qwen_chunks.jsonl").exists():
        return p / "outputs"
    if (p / "qwen_chunks.jsonl").exists():
        return p
    raise FileNotFoundError(
        f"RUN_ID_OR_OUTPUT_DIR={value!r} did not resolve to qwen_chunks.jsonl. Tried: {p}"
    )

OUT_DIR = _candidate_from_user_value(RUN_ID_OR_OUTPUT_DIR)

if OUT_DIR is None:
    qwen_files = []
    if RUNS_ROOT.exists():
        qwen_files.extend(RUNS_ROOT.glob("smoke_*/outputs/qwen_chunks.jsonl"))
        qwen_files.extend(RUNS_ROOT.glob("*/outputs/qwen_chunks.jsonl"))
    elif AUTO_DISCOVER_UNDER_MYDRIVE:
        print(f"Default RUNS_ROOT not found: {RUNS_ROOT}")

    if not qwen_files and AUTO_DISCOVER_UNDER_MYDRIVE:
        print("Searching MyDrive for qwen_chunks.jsonl. This may take a minute...")
        qwen_files = list(my_drive.rglob("qwen_chunks.jsonl"))

    # Prefer smoke outputs, then newest by mtime.
    qwen_files = sorted(set(qwen_files), key=lambda f: ("smoke_" not in str(f), -f.stat().st_mtime))
    if not qwen_files:
        print("Could not find qwen_chunks.jsonl anywhere under expected paths.")
        print("Quick Drive listing:")
        for root in [my_drive, my_drive / "asr", my_drive / "asr" / "qwen-asr-eval"]:
            print("\n--", root)
            if root.exists():
                for child in sorted(root.iterdir())[:50]:
                    print(child)
            else:
                print("missing")
        raise FileNotFoundError(
            "No qwen_chunks.jsonl found. The CLI smoke script may have run under a different Google account, "
            "or it did not complete ASR/output writing. Paste the actual outputs directory into RUN_ID_OR_OUTPUT_DIR."
        )
    OUT_DIR = qwen_files[0].parent

LATEST_RUN = OUT_DIR.parent if OUT_DIR.name == "outputs" else OUT_DIR
print("REPO_DIR:", REPO_DIR)
print("RUNS_ROOT:", RUNS_ROOT, "exists=", RUNS_ROOT.exists())
print("LATEST_RUN:", LATEST_RUN)
print("OUT_DIR:", OUT_DIR)
print("qwen_chunks:", OUT_DIR / "qwen_chunks.jsonl")
print("Files:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path.name)


In [ ]:
#@title Load Qwen rows from the script output
import json
import pandas as pd

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

qwen_rows = read_jsonl(OUT_DIR / "qwen_chunks.jsonl")
whisper_path = OUT_DIR / "whisper_chunks.jsonl"
whisper_rows = read_jsonl(whisper_path) if whisper_path.exists() else None

USE_CASE = "mixed"  #@param ["auto", "zh", "en", "mixed"]
COLAB_AI_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
MAX_TEXT_JUDGE_CHUNKS = 1  #@param {type:"integer"}
RUN_COLAB_AI_TEXT_JUDGE = True

print("qwen rows:", len(qwen_rows))
print("whisper rows:", 0 if whisper_rows is None else len(whisper_rows))
print("preview:")
print(qwen_rows[0].get("text", "")[:1200])
display(pd.DataFrame(qwen_rows)[["chunk_id", "start_ts", "end_ts", "detected_language", "text"]].head())


In [ ]:
#@title Check Colab AI availability in the web UI
from google.colab import ai

available_models = ai.list_models()
print("Available Colab AI models:")
for model in available_models:
    print("-", model)

if COLAB_AI_MODEL not in available_models:
    print(f"Configured COLAB_AI_MODEL={COLAB_AI_MODEL!r} not found; falling back to first available model.")
    COLAB_AI_MODEL = available_models[0]
print("Using:", COLAB_AI_MODEL)


In [ ]:
#@title Run Colab AI text-only triage for the script output
from asr_eval.colab_ai_judge import judge_chunks_colab_ai, select_eval_chunk_ids
import json
import pandas as pd

if RUN_COLAB_AI_TEXT_JUDGE:
    eval_chunk_ids = select_eval_chunk_ids(qwen_rows, whisper_rows, max_chunks=MAX_TEXT_JUDGE_CHUNKS)
    print("Selected chunk IDs:", eval_chunk_ids)
    colab_ai_reports = judge_chunks_colab_ai(
        qwen_rows,
        whisper_rows,
        use_case=USE_CASE,
        model_name=COLAB_AI_MODEL,
        max_chunks=MAX_TEXT_JUDGE_CHUNKS,
    )
    judge_path = OUT_DIR / "colab_ai_text_judge.json"
    judge_path.write_text(json.dumps(colab_ai_reports, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", judge_path)
    display(pd.DataFrame(colab_ai_reports))
else:
    colab_ai_reports = []
    print("RUN_COLAB_AI_TEXT_JUDGE=False; skipping.")


## 9. Optional text-only triage with Colab AI

This uses `google.colab.ai.generate_text`. It is free/config-light inside Colab and does **not** need a Gemini API key.

It cannot hear audio. It only inspects Qwen/Whisper transcript text and flags suspicious chunks for manual review.

In [ ]:
#@title List available Colab AI models
if RUN_COLAB_AI_TEXT_JUDGE:
    try:
        from google.colab import ai
        available_models = ai.list_models()
        print("Available Colab AI models:")
        for m in available_models:
            print("-", m)
        if COLAB_AI_MODEL not in available_models:
            print(f"Configured COLAB_AI_MODEL={COLAB_AI_MODEL!r} not found; falling back to first available model.")
            COLAB_AI_MODEL = available_models[0]
        print("Using:", COLAB_AI_MODEL)
    except Exception as e:
        print("Colab AI unavailable:", repr(e))
        RUN_COLAB_AI_TEXT_JUDGE = False
else:
    print("RUN_COLAB_AI_TEXT_JUDGE=False")

In [ ]:
#@title Run Colab AI text-only ASR output triage
from asr_eval.colab_ai_judge import judge_chunks_colab_ai, select_eval_chunk_ids
import json, pandas as pd

if RUN_COLAB_AI_TEXT_JUDGE:
    eval_chunk_ids = select_eval_chunk_ids(qwen_rows, whisper_rows, max_chunks=MAX_TEXT_JUDGE_CHUNKS)
    print("Selected chunk IDs:", eval_chunk_ids)
    colab_ai_reports = judge_chunks_colab_ai(
        qwen_rows,
        whisper_rows,
        use_case=USE_CASE,
        model_name=COLAB_AI_MODEL,
        max_chunks=MAX_TEXT_JUDGE_CHUNKS,
    )
    judge_path = OUT_DIR / "colab_ai_text_judge.json"
    judge_path.write_text(json.dumps(colab_ai_reports, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", judge_path)
    display(pd.DataFrame(colab_ai_reports))
else:
    colab_ai_reports = []
    print("Skipping Colab AI text-only triage.")

## 10. Output summary

Recommended first-pass interpretation:

- Trust reference WER/CER most, if you have reference transcript.
- Use Whisper as a baseline to find disagreement chunks.
- Use Colab AI text-only judge only as a triage assistant.
- Manually listen to flagged chunks before making a quality verdict.

In [ ]:
#@title List output files
print("Run dir:", RUN_DIR)
print("Output dir:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(f"{p.name:40s} {p.stat().st_size/1024:.1f} KiB")